# Embedding Model using `sentence_transformer`

In this notebook, we explore the way that text with multiple words/tokens is encoded into an embedding vector using the popular `sentence_transformer` library.

We will check:
- OpenAI Embedding
- Open source encoder input embeddings
- Open source encoder output embedding (with context)
- Improved encoder for queries and documents (bi-encoder)

In [3]:
# Define rich theme for better print results
from rich.console import Console
from rich_theme_manager import Theme, ThemeManager
import pathlib

theme_dir = pathlib.Path("themes")
theme_manager = ThemeManager(theme_dir = theme_dir)
dark = theme_manager.get("dark")

# Create a console with dark theme
console = Console(theme = dark)

In [4]:
import warnings

# Suppress warnings
warnings.filterwarnings("ignore")

## OpenAI Embedding
A common option is to use the embedding from the same provider as the generation model.

In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
first_sentence = "I have no interest in politics"

In [10]:
from openai import OpenAI
client = OpenAI()
response = client.embeddings.create(
    input=first_sentence,
    model="text-embedding-3-small"
)

# console.print(response)

In [ ]:
# Create a preview of the response with truncated embedding for better readability
preview = response.model_copy(deep=True)
preview.data[0].embedding = f"{preview.data[0].embedding[:3]}...{preview.data[0].embedding[-3:]}"
console.print(preview)


CreateEmbeddingResponse(
    data=[
        Embedding(
            embedding='[-0.03131103515625, -0.0022525787353515625, -0.0244903564453125]...[0.012664794921875, 
0.01528167724609375, 0.0208282470703125]',
            index=0,
            object='embedding'
        )
    ],
    model='text-embedding-3-small',
    object='list',
    usage=Usage(prompt_tokens=6, total_tokens=6)
)

## Open source encoder = input embeddings
We will start with popular encoders from the `sentence_transformers` library.

It will allow us to explore its architecture and flow, and later on to optimize it to our use-case

In [15]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

### Model Tokenizer
We will use the default tokenizer of the model.

Every word or sub-word is converted into a token with a constant ID.

E.g., in the following two sentences: the word `interest` is tokenized to the same ID.

In [16]:
first_sentence = "I have no interest in politics"
second_sentence = "The bank's interest rate rises"

In [17]:
tokenized_first_sentence = model.tokenize([first_sentence])
console.rule(f"{first_sentence}")
console.print(tokenized_first_sentence)

───────────────────────────────────────── I have no interest in politics ──────────────────────────────────────────

{
    'input_ids': tensor([[ 101, 1045, 2031, 2053, 3037, 1999, 4331,  102]]),
    'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]),
    'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])
}

In [18]:
tokenized_second_sentence = model.tokenize([second_sentence])
console.rule(f"{second_sentence}")
console.print(tokenized_second_sentence)

───────────────────────────────────────── The bank's interest rate rises ──────────────────────────────────────────

{
    'input_ids': tensor([[ 101, 1996, 2924, 1005, 1055, 3037, 3446, 9466,  102]]),
    'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0]]),
    'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])
}

The word `interest` in both sentences is tokenised to `3037`

The token ID can be used to convert it back into readable text.

In [19]:
sentence_tokens = (
    model
    .tokenizer
    .convert_ids_to_tokens(
        tokenized_first_sentence["input_ids"][0]
    )
)

console.print(sentence_tokens)

['[CLS]', 'i', 'have', 'no', 'interest', 'in', 'politics', '[SEP]']

### Model Vocabulary
We will see how many words the model knows in total and preview few of those.

In [22]:
vocabulary = (
    model
    ._first_module()
    .tokenizer
    .get_vocab()
    .items()
)

console.print("[bold] Vocabulary Size: [/bold]", len(vocabulary))
console.print(dict(list(vocabulary)[:10]))

 Vocabulary Size:  30522

{
    'die': 3280,
    '[unused394]': 399,
    '##room': 9954,
    '##haven': 14650,
    'ᅴ': 1482,
    'linguistics': 15397,
    '##mei': 26432,
    'cheap': 10036,
    '##kers': 11451,
    'bubble': 11957
}

We will search for the token for interest and see its neighbors.

In [25]:
sorted_vocabulary = sorted(
    vocabulary,
    key=lambda x: x[1] # Uses the value of the dictionary entry
)

sorted_tokens = [token for token, _ in sorted_vocabulary]

console.print(sorted_vocabulary[:10])
console.print("total length of sorted vocabulary: ", len(sorted_vocabulary))
console.print(sorted_tokens[:10])
console.print("total length of sorted tokens: ", len(sorted_tokens))


[
    ('[PAD]', 0),
    ('[unused0]', 1),
    ('[unused1]', 2),
    ('[unused2]', 3),
    ('[unused3]', 4),
    ('[unused4]', 5),
    ('[unused5]', 6),
    ('[unused6]', 7),
    ('[unused7]', 8),
    ('[unused8]', 9)
]

total length of sorted vocabulary:  30522

[
    '[PAD]',
    '[unused0]',
    '[unused1]',
    '[unused2]',
    '[unused3]',
    '[unused4]',
    '[unused5]',
    '[unused6]',
    '[unused7]',
    '[unused8]'
]

total length of sorted tokens:  30522

In [28]:
focused_token = 'interest'

# find the index of the 'interest' token
focused_index = sorted_tokens.index(focused_token)

console.print("Index of the word 'interest' was found to be:", focused_index)

Index of the word 'interest' was found to be: 3037

Get 20 tokens nearest to the focused token.

In [32]:
start_index = max(0, focused_index - 10)
end_index = min(len(sorted_tokens), focused_index + 11)
tokens_around_focused_index = sorted_tokens[start_index:end_index]

# console.print("Tokens around 'interest':", tokens_around_focused_index)

from rich.table import Table

table = Table(title=f"Tokens around '{focused_token}':")
table.add_column("id", justify="right", style="cyan", no_wrap=True)
table.add_column("token", style="bright_green")

for i, token in enumerate(tokens_around_focused_index, start=start_index):
    if token == focused_token:
        table.add_row(f"[bold][black on yellow]{i}[/black on yellow][/bold]", f"[bold][black on yellow]{token}[/black on yellow][/bold]")
    else:
        table.add_row(str(i), token)

console.print(table)

     Tokens around     
      'interest':      
┏━━━━━━┳━━━━━━━━━━━━━━┓
┃   id ┃ token        ┃
┡━━━━━━╇━━━━━━━━━━━━━━┩
│ 3027 │ ft           │
│ 3028 │ valley       │
│ 3029 │ organization │
│ 3030 │ stopped      │
│ 3031 │ onto         │
│ 3032 │ countries    │
│ 3033 │ parts        │
│ 3034 │ conference   │
│ 3035 │ queen        │
│ 3036 │ security     │
│ 3037 │ interest     │
│ 3038 │ saying       │
│ 3039 │ allowed      │
│ 3040 │ master       │
│ 3041 │ earlier      │
│ 3042 │ phone        │
│ 3043 │ matter       │
│ 3044 │ smith        │
│ 3045 │ winning      │
│ 3046 │ try          │
│ 3047 │ happened     │
└──────┴──────────────┘